# STEP-047 — Cross-dataset evaluation of MODEL0001 (thin caller)

All logic lives in `17_Automation/generalization`; target datasets live in `08_Evaluation/EVAL0001/generalization/config.yaml`. Do not add code or parameters here. Runbook: `08_Evaluation/EVAL0001/generalization/KAGGLE_RUNBOOK.md`.

**Notebook settings:** Accelerator `GPU T4 x2`, Internet `On`, secret `GITHUB_TOKEN`. Inputs: `isthisdeception/ds0001-pp0001`, `ds0002-pp0002`, `ds0003-pp0003`, `ds0004-pp0006`, `ds0005-pp0005`, `thesis-experiment-artifacts`.

Run every cell in order: 0 Setup → A Inference → B Analysis → C Push. If interrupted, run 0 and A again: finished datasets are reused.

## 0. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/isthisdeception/thesis.git'
BRANCH = 'main'
REPO = Path('/kaggle/working/thesis')
PLAN = 'kaggle_sync/sync_plan.generalization.yaml'
AUTOMATION = REPO / '17_Automation'
OUT = REPO / '08_Evaluation' / 'EVAL0001' / 'generalization'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# GITHUB_TOKEN (Add-ons -> Secrets, ticked for this notebook) is needed only for C.
try:
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('GITHUB_TOKEN loaded')
    except Exception:
        print('GITHUB_TOKEN not set: C will fail')
except ImportError:
    print('kaggle_secrets unavailable (not on Kaggle)')

In [ ]:
# Pull latest main, install environment/kaggle-requirements.txt, verify all six inputs are attached.
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} bootstrap

In [ ]:
import torch, fastai
from importlib.metadata import version
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
print('fastai', fastai.__version__, '| plum-dispatch', version('plum-dispatch'))
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'GPU not visible: set Accelerator to GPU T4 x2'
!cd {AUTOMATION} && python -m generalization datasets

## A. Inference (about 1 hour; most of it is reading images)

Expect `export: ...MODEL0001_export.pkl`, then `sanity check PASS`, then one line per dataset. If it stops with `CleanCheckError`, do not continue; report the message. Re-running this cell skips finished datasets.

In [ ]:
!cd {AUTOMATION} && python -m generalization run

## B. Analysis (under a minute)

In [ ]:
!cd {AUTOMATION} && python -m generalization analyze
import pandas as pd
pd.set_option('display.width', 200)
print(pd.read_csv(OUT / 'cross_dataset_metrics.csv')[['dataset', 'n', 'n_fake', 'n_real', 'recall', 'specificity', 'balanced_accuracy', 'roc_auc']].to_string(index=False))

## C. Push results to GitHub (needs GITHUB_TOKEN)

Pushes only `08_Evaluation/EVAL0001/generalization` to a `kaggle/` branch; it reaches `main` through a reviewed PR.

In [ ]:
from datetime import datetime
PUSH_BRANCH = f"kaggle/cross-dataset-{datetime.now():%Y%m%d-%H%M}"
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} push-metadata --branch {PUSH_BRANCH} --commit-message "sync: EVAL0001 cross-dataset results from Kaggle"
print('pushed to', PUSH_BRANCH)